#Sampling Activities from Flyvision

This notebook is adapted from the published notebook at https://turagalab.github.io/flyvis/examples/07_flyvision_providing_custom_stimuli/

# Providing custom stimuli

Follow this notebook to learn how to use our models for generating hypothesis about neural computations with custom stimuli.

## Setup (one-time)

Install `flyvis` and download pretrained models before running this notebook:

```bash
pip install flyvis[examples]
flyvis download-pretrained
```

The pretrained models (~500 MB) are cached in the flyvis data directory and only need to be downloaded once.

In [ ]:
import flyvis, os

if not (flyvis.results_dir / "flow/0000").exists():
    print("Pretrained models not found. Downloading...")
    os.system("flyvis download-pretrained")
else:
    print("Pretrained models found:", flyvis.results_dir / "flow/0000")

In [ ]:
# ============================================================
# CONFIG — Edit this section before running
# ============================================================

# --- Cell type group to sample ---
# Set GROUP_NAME to one of the keys in CELL_TYPE_GROUPS, or set CELL_TYPES manually.
GROUP_NAME = "Medulla"   # used in output filename

CELL_TYPE_GROUPS = {
    "Retina":  ["R1", "R2", "R3", "R4", "R5", "R6", "R7", "R8"],
    "Lamina":  ["L1", "L2", "L3", "L4", "L5", "Am", "C2", "C3", "CT1(Lo1)", "CT1(M10)"],
    "Medulla": ["Mi1", "Mi2", "Mi3", "Mi4", "Mi9", "Mi10", "Mi11", "Mi12",
                "Mi13", "Mi14", "Mi15"],
    "T_Tm":   ["T1", "T2", "T2a", "T3", "T4a", "T4b", "T4c", "T4d",
               "T5a", "T5b", "T5c", "T5d",
               "Tm1", "Tm2", "Tm3", "Tm4", "Tm5Y", "Tm5a", "Tm5b", "Tm5c",
               "Tm9", "Tm16", "Tm20", "Tm28", "Tm30",
               "TmY3", "TmY4", "TmY5a", "TmY9", "TmY10", "TmY13", "TmY14", "TmY15", "TmY18"],
    "All":    ["R1", "R2", "R3", "R4", "R5", "R6", "R7", "R8",
               "L1", "L2", "L3", "L4", "L5", "Am", "C2", "C3", "CT1(Lo1)", "CT1(M10)",
               "Mi1", "Mi2", "Mi3", "Mi4", "Mi9", "Mi10", "Mi11", "Mi12",
               "Mi13", "Mi14", "Mi15",
               "T1", "T2", "T2a", "T3", "T4a", "T4b", "T4c", "T4d",
               "T5a", "T5b", "T5c", "T5d",
               "Tm1", "Tm2", "Tm3", "Tm4", "Tm5Y", "Tm5a", "Tm5b", "Tm5c",
               "Tm9", "Tm16", "Tm20", "Tm28", "Tm30",
               "TmY3", "TmY4", "TmY5a", "TmY9", "TmY10", "TmY13", "TmY14", "TmY15", "TmY18"],
}

CELL_TYPES = CELL_TYPE_GROUPS[GROUP_NAME]

# --- Model(s) ---
MODEL_IDXS = ["000"]   # list of pretrained model indices to run and average; "000" = best task error

# --- Stimulus ---
topdir = '../stimuli/flowstims'  # path to flowstims directory (relative to notebook)
scl_factor = 0.5          # scale factor applied to orig_shape before center-cropping
N_INSTANCES = 3           # number of stimulus instances (different random seeds)
trial_len = 75 // 2       # frames per stimulus trial (= 37)
stride = 1
orig_shape = (800, 600)   # original flowstim image size (width, height)
input_shape = (48, 48)    # crop size fed to BoxEye (height, width)

# --- Neuron sampling ---
seed = 0
N_FMAPS_TO_SAMPLE = 40   # max number of cell types to sample
SAMPLES_PER_FMAP = 50    # max receptors (neurons) to sample per cell type
fmap_samp_method = 'maxFr'
neur_samp_method = 'maxNr'
LAYER_TYPE = 'act'

print(f"GROUP_NAME: {GROUP_NAME}  ({len(CELL_TYPES)} cell types)")
print(f"MODEL_IDXS: {MODEL_IDXS}")
print(f"CELL_TYPES: {CELL_TYPES}")

## Utils

In [ ]:
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from glob import glob
from scipy.io import loadmat

def createFlowDataset(categories, topdir, mydirs, orig_shape, input_shape, scl_factor, N_INSTANCES, trial_len, stride):
    scld_shape = tuple((np.array(orig_shape)*scl_factor).astype('int'))
    NDIRS = len(mydirs)
    frames_per_stim = int(np.ceil(trial_len/stride))

    shift_foos = {'0':lambda im,step: np.roll(im,step,1),
                  '45':lambda im,step: np.roll(np.roll(im,step,1),-step,0),
                  '90':lambda im,step: np.roll(im,-step,0),
                  '135':lambda im,step: np.roll(np.roll(im,-step,1),-step,0),
                  '180':lambda im,step: np.roll(im,-step,1),
                  '225':lambda im,step: np.roll(np.roll(im,-step,1),step,0),
                  '270':lambda im,step: np.roll(im,step,0),
                  '315':lambda im,step: np.roll(np.roll(im,step,0),step,1),
                 }

    flow_datasets = {}

    for inst_i in range(N_INSTANCES):
        print('*INSTANCE',inst_i,end=' ',flush=True)
        for cat in categories:
            print('.',end='',flush=True)
            stim_arrays = None

            for di,d in enumerate(mydirs):

                image_path = f'{topdir}/{cat}_inst{inst_i}/{d}/0.png'
                img = Image.open(image_path)

                assert orig_shape == img.size

                if scl_factor != 1:
                    img = img.resize(scld_shape, Image.Resampling.LANCZOS)

                #cropping idxs
                w,h = img.size
                assert w == scld_shape[0] and h == scld_shape[1]
                i0, j0 = h//2-input_shape[0]//2, w//2-input_shape[1]//2
                i1, j1 = i0 + input_shape[0], j0 + input_shape[1]

                img_array = np.array(img)[:,:,0] #since grayscale, use only one channel

                for fii,fi in enumerate(range(0,trial_len,stride)):
                    #shift full img
                    shifted_img = shift_foos[d](img_array,fi)
                    #crop from center
                    shifted_img = shifted_img[i0:i1,j0:j1]
                    #save
                    if stim_arrays is None:
                        stim_arrays = np.zeros((NDIRS*frames_per_stim,shifted_img.size))
                    stim_arrays[di*frames_per_stim+fii] = shifted_img.ravel()


            if inst_i not in flow_datasets:
                flow_datasets[inst_i] = stim_arrays
            else:
                flow_datasets[inst_i] = np.concatenate([flow_datasets[inst_i],stim_arrays])

        print()
    return flow_datasets

def from0to1(arr):
    arr = np.asanyarray(arr)
    arr[np.isclose(arr,0)] = 1
    return arr

def subps(nrows,ncols,rowsz=3,colsz=4,d3=False,axlist=False):
    if d3:
        f = plt.figure(figsize=(ncols*colsz,nrows*rowsz))
        axes = [[f.add_subplot(nrows,ncols,ri*ncols+ci+1, projection='3d') for ci in range(ncols)] \
                for ri in range(nrows)]
        if nrows == 1:
            axes = axes[0]
            if ncols == 1:
                axes = axes[0]
    else:
        f,axes = plt.subplots(nrows,ncols,figsize=(ncols*colsz,nrows*rowsz))
    if axlist and ncols*nrows == 1:

        axes = [axes]
    return f,axes

## Stimulus generation

Load the optical flow stimuli and preview a few frames. Stimuli include gratings at three spatial frequencies and dot flows at two speeds/densities, each presented in 8 directions (0°–315°). All parameters come from the CONFIG cell.

In [ ]:
import os
import numpy as np
import torch
import matplotlib.pyplot as plt
from PIL import Image
from glob import glob
from time import time
from tqdm import tqdm

import flyvis
from flyvis.datasets.rendering import BoxEye
from flyvis.analysis import animations
from flyvis.analysis.visualization import plt_utils, plots

In [ ]:
mydirs = list(map(str, range(0, 360, 45)))
categories = ['grat_W12', 'grat_W1', 'grat_W2',
              'neg1dotflow_D1_bg', 'neg3dotflow_D1_bg', 'neg1dotflow_D2_bg', 'neg3dotflow_D2_bg',
              'pos1dotflow_D1_bg', 'pos3dotflow_D1_bg', 'pos1dotflow_D2_bg', 'pos3dotflow_D2_bg']

NDIRS = len(mydirs)
tot_stims = len(categories) * NDIRS
frames_per_stim = trial_len // stride
print(f'categories: {len(categories)},  directions: {NDIRS},  tot_stims: {tot_stims},  frames_per_stim: {frames_per_stim}')

flow_datasets = createFlowDataset(categories, topdir, mydirs, orig_shape, input_shape,
                                  scl_factor, N_INSTANCES, trial_len, stride)

# Show example frames from instance 0
n_frames_to_show = 4
interval = 8
f, axes = subps(1, n_frames_to_show, 1, 1)
for i in range(n_frames_to_show):
    ax = axes[i]
    img = flow_datasets[0][i * interval].reshape(input_shape)
    ax.imshow(img, vmin=0, vmax=255, cmap='gray')
    ax.axis('off')
f.tight_layout()
plt.show()

In [ ]:
print('flow_datasets[0].shape:', flow_datasets[0].shape)
sequences = flow_datasets[0].reshape(-1, frames_per_stim, input_shape[0], input_shape[1]) / 255
print('sequences.shape:', sequences.shape)

In [ ]:
animation = animations.Imshow(sequences, cmap=plt.cm.binary_r)
animation.animate_in_notebook(samples=[0, 1, 2])

Alternative: for an alternative dataset that is generated at runtime and does not require a download try `random_walk_of_blocks`. As a simple drop-in replacement, this requires to replace `load_moving_mnist` with `random_walk_of_blocks` across the notebook.

## BoxEye rendering

##### Rendering cartesian images to hexagonal lattice

We translate cartesian frames into receptor activations by placing simulated photoreceptors in a two-dimensional hexagonal array in pixel space (blue dots below), 31 columns across resulting in 721 columns in total, spaced 13 pixels apart. The transduced luminance at each photoreceptor is the greyscale mean value in the 13×13-pixel region surrounding it (black boxes).

In [ ]:
import flyvis
from flyvis.datasets.rendering import BoxEye

In [ ]:
receptors = BoxEye(extent=15, kernel_size=13)

In [ ]:
fig = receptors.illustrate()

### Render a single frame

To illustrate, this is what rendering a single frame looks like.

In [ ]:
plt.rcParams['figure.dpi'] = 200

In [ ]:
fig, ax = plt_utils.init_plot(figsize=[1, 1], fontsize=5)
ax = plt_utils.rm_spines(ax)
ax.imshow(sequences[0, 0], cmap=plt.cm.binary_r)
_ = ax.set_title('example frame', fontsize=5)

In [ ]:
single_frame = sequences[20, 0]

# the rendering uses pytorch native Conv2d module so it can be executed on GPU and fast
# we first move the frame to GPU
single_frame = torch.tensor(single_frame, device=flyvis.device).float()

# because the inputs to the receptors instance must have four dimensions (samples, frames, height, width),
# we create two empty dimensions for samples and frames
single_frame = single_frame[None, None]

In [ ]:
# to render the single frame we simply call the instance
# this automatically rescales the frame to match the receptor layout as illustrated above
# and then places the average pixel value of the 13x13 boxes at the receptor positions
receptors = BoxEye()
rendered = receptors(single_frame)

In [ ]:
# the 721 receptor coordinates are implicitly given in the last dimension
# they correspond to sorted hexagonal coordinates (u-coordinate, v-coordinate, value)
rendered.shape

In [ ]:
# the rendered frame is a slightly blurred version of the example
fig, ax, _ = plots.quick_hex_scatter(
    rendered.squeeze(), vmin=0, vmax=1, cbar_x_offset=0, fontsize=5
)
_ = ax.set_title("example frame rendered", fontsize=5)

## Compute model responses

Run the flyvis network on the rendered stimuli. Responses are computed for each cell type in `CELL_TYPES` (from CONFIG) across all `N_INSTANCES` instances and averaged. Rendering is done in-memory per instance (avoids stale caching across instances).

## Compute model responses to custom stimuli

Now, we can compute model responses across individual models or the whole ensemble to our custom stimulus.

##### Select a pretrained network

To select a network from the ensemble of 50 pretrained networks, let's see what our options are.

Paths to pretrained models from the ensemble end with four digit numbers which are sorted by task error (0-49 from best to worst).

In [ ]:
sorted([
    p.relative_to(flyvis.results_dir)
    for p in (flyvis.results_dir / "flow/0000").iterdir()
    if p.name.isnumeric()
])

We use the `NetworkView` class to point to a model. This object can implement plots plus methods to initialize network, stimuli etc.

In [ ]:
network_view = flyvis.NetworkView(flyvis.results_dir / f"flow/0000/{MODEL_IDXS[0]}")

In [ ]:
# to load the Pytorch module with pretrained parameters
network = network_view.init_network()

In [ ]:
####################### COMPUTE MODEL RESPONSES ################

layers_to_use = CELL_TYPES   # alias used throughout (from CONFIG)

n_orig_imgs = tot_stims
n_shifts = frames_per_stim
n_shifted_imgs = n_orig_imgs * n_shifts
print(f'Stimuli: {n_orig_imgs} × {n_shifts} frames = {n_shifted_imgs} total')
print(f'Cell types ({len(layers_to_use)}): {layers_to_use}')
print(f'Models: {MODEL_IDXS}')

receptors = BoxEye(extent=15, kernel_size=13)
DT = 1 / 100  # model integration timestep (seconds)

from flyvis.utils.activity_utils import LayerActivity

# Lazily initialized on first response; actual size depends on cell type neuron count.
n_neurons_per_type = [None] * len(layers_to_use)
layer_outputs = [None] * len(layers_to_use)

total_runs = 0
for model_idx in MODEL_IDXS:
    _network_view = flyvis.NetworkView(flyvis.results_dir / f"flow/0000/{model_idx}")
    network = _network_view.init_network()

    for insti in range(N_INSTANCES):
        extX = flow_datasets[insti]
        print(f'\n--- MODEL {model_idx}  INSTANCE {insti} ---  raw stim shape: {extX.shape}')
        assert extX.shape[0] == n_shifted_imgs

        # Render stimuli to hexagonal receptor activations in memory.
        # Done per instance to correctly use each instance's unique stimulus images.
        sequences_arr = extX.reshape(-1, frames_per_stim, input_shape[0], input_shape[1]) / 255.0
        rendered_list = []
        for idx in tqdm(range(sequences_arr.shape[0]), desc='Rendering to hex lattice'):
            frame_tensor = torch.tensor(sequences_arr[[idx]], device=flyvis.device).float()
            rendered_list.append(receptors(frame_tensor).cpu().numpy())
        rendered = np.concatenate(rendered_list, axis=0)  # (n_orig_imgs, frames_per_stim, 1, hexals)
        print(f'Rendered shape: {rendered.shape}')

        layer_output = [[] for _ in range(len(layers_to_use))]
        start0 = time()

        for seq_idx in tqdm(range(n_orig_imgs), desc=f'Simulating model {model_idx} inst {insti}'):
            movie_input = torch.tensor(rendered[seq_idx], device=flyvis.device).float()  # (n_frames, 1, hexals)
            stationary_state = network.fade_in_state(1.0, DT, movie_input[[0]])
            output = network.simulate(movie_input[None], DT, initial_state=stationary_state).cpu()

            responses = LayerActivity(output, network.connectome, keepref=True)
            for li in range(len(layers_to_use)):
                resp = responses[layers_to_use[li]][:, :, None]  # (batch, n_frames, 1, n_cells)
                layer_output[li].append(np.array(resp))

        print(f'Simulation time: {time() - start0:.1f}s')

        for li in range(len(layers_to_use)):
            arr = np.concatenate(layer_output[li], axis=0)   # (n_orig_imgs, n_frames, 1, n_cells)
            n_cells = arr.shape[-1]
            arr = arr.reshape((-1, n_cells, 1, 1))            # (n_shifted_imgs, n_cells, 1, 1)
            # Lazy init: allocate using the actual neuron count from the first response.
            if layer_outputs[li] is None:
                n_neurons_per_type[li] = n_cells
                layer_outputs[li] = np.zeros([n_shifted_imgs, n_cells, 1, 1], dtype='float32')
            layer_outputs[li] += arr

        total_runs += 1

# Average over all models × instances
for li in range(len(layers_to_use)):
    layer_outputs[li] /= total_runs

print(f'\nDone. Averaged over {total_runs} runs ({len(MODEL_IDXS)} model(s) × {N_INSTANCES} instance(s)).')
print(f'layer_outputs[0].shape: {layer_outputs[0].shape}')  # (n_shifted_imgs, n_cells, 1, 1)
print(f'n_neurons_per_type: {dict(zip(layers_to_use, n_neurons_per_type))}')

In [ ]:
################### SUMMARIZE ACTIVITY ###########
# For each cell type, compute per-stimulus max and mean activity across receptors and time.
# Results are concatenated into all_per_img_output: (n_stims, n_cell_types, n_receptors, n_frames)

print('Computing per-stimulus statistics...')
all_neurons_maxs = None
all_neurons_means = None
all_per_img_output = None

for li in range(len(layers_to_use)):
    layer_output_ = layer_outputs[li].copy()

    # Use absolute value so that strongly inhibitory neurons are selected on equal footing
    # with excitatory ones. The raw (signed) responses are preserved in all_per_img_output.
    layer_output_abs = np.abs(layer_output_)

    # nfmaps = 1 per cell type: shape is (n_shifted_imgs, n_cells, 1, 1)
    nfmaps = layer_output_.shape[3]  # = 1

    # Store raw responses (signed) for the output tensor
    orig_per_img_output = np.moveaxis(layer_output_, -1, 1).reshape([n_orig_imgs, n_shifts, nfmaps, -1])
    orig_per_img_output = np.moveaxis(orig_per_img_output, 1, -1)  # (n_stims, nfmaps, n_receptors, n_frames)

    # Normalize absolute responses by per-image max for statistics
    layer_output_abs /= np.maximum(layer_output_abs.max((1, 2, 3), keepdims=True), 1e-8)
    per_img_output = np.moveaxis(layer_output_abs, -1, 1).reshape([n_orig_imgs, n_shifts, nfmaps, -1])
    per_img_output = np.moveaxis(per_img_output, 1, -1)

    neurons_maxs = np.zeros(per_img_output.shape[1:3])
    neurons_means = np.zeros(per_img_output.shape[1:3])
    for imi in range(n_orig_imgs):
        im_avgs = per_img_output[imi].mean(2)  # average across time -> (nfmaps, n_receptors)
        neurons_maxs = np.maximum(neurons_maxs, im_avgs)
        neurons_means += im_avgs
    neurons_means /= n_orig_imgs

    if all_neurons_maxs is None:
        all_neurons_maxs = neurons_maxs
        all_neurons_means = neurons_means
        all_per_img_output = orig_per_img_output
    else:
        all_neurons_maxs = np.concatenate([all_neurons_maxs, neurons_maxs], axis=0)
        all_neurons_means = np.concatenate([all_neurons_means, neurons_means], axis=0)
        all_per_img_output = np.concatenate([all_per_img_output, orig_per_img_output], axis=1)

print(f'all_neurons_maxs:   {all_neurons_maxs.shape}   (n_cell_types, n_receptors_per_type)')
print(f'all_per_img_output: {all_per_img_output.shape}  (n_stims, n_cell_types, n_receptors, n_frames)')

In [ ]:
############# SAMPLE NEURONS ###########
np.random.seed(seed)

nfmaps, n_neurons_per_fmap = all_neurons_maxs.shape  # (n_cell_types, 721)

# Each cell type contributes exactly 1 "fmap"; fi is directly the cell type index.
layer_is_per_fmap = np.arange(len(layers_to_use))  # [0, 1, 2, ..., n_cell_types-1]

maxsmean = all_neurons_maxs.mean(1)
nonzero_indices = (~np.isclose(maxsmean, 0)).sum()
n_fmaps_to_sample_ = min(N_FMAPS_TO_SAMPLE, nonzero_indices)
print(f'Sampling {n_fmaps_to_sample_} / {nfmaps} cell types  ({nonzero_indices} non-zero)')

if fmap_samp_method == 'maxFr':
    probabilities = maxsmean / maxsmean.sum()
    top_fmaps = np.random.choice(range(nfmaps), n_fmaps_to_sample_, replace=False, p=probabilities)
else:
    raise ValueError(f'Unknown fmap_samp_method: {fmap_samp_method}')

samples_per_fmap_ = min(SAMPLES_PER_FMAP, n_neurons_per_fmap)
print(f'Sampling up to {samples_per_fmap_} / {n_neurons_per_fmap} receptors per cell type')

sampled_neurons = []
for fi in top_fmaps:
    if neur_samp_method == 'maxNr':
        neuron_vals = all_neurons_maxs[fi]
        nonzero_neurons = (~np.isclose(neuron_vals, 0)).sum()
        n_to_pick = min(samples_per_fmap_, nonzero_neurons)
        probabilities = neuron_vals / neuron_vals.sum()
        top_nis = np.random.choice(range(n_neurons_per_fmap), n_to_pick, replace=False, p=probabilities)
    else:
        raise ValueError(f'Unknown neur_samp_method: {neur_samp_method}')
    sampled_neurons += list(fi * n_neurons_per_fmap + top_nis)

sampled_neurons = np.array(sampled_neurons)
n_neurons_to_pick = len(sampled_neurons)
print(f'Total neurons sampled: {n_neurons_to_pick}')

In [ ]:
######### BUILD TENSOR + SAVE ##########

def get_neuron_pos(ni):
    """Return (cell_type_idx, cell_type_idx, receptor_i, 0, receptor_idx) for flat index ni.

    Each cell type has n_neurons_per_fmap receptors arranged as (h=n_neurons_per_fmap, w=1).
    neurons_used columns: [layer_idx, fmap_idx, i, j, receptor_pos]
    """
    fi = ni // n_neurons_per_fmap    # cell type index
    li = int(layer_is_per_fmap[fi])  # same as fi for flyvis (1 fmap per cell type)
    posi = ni % n_neurons_per_fmap   # receptor index
    h, w = n_neurons_per_fmap, 1
    ii = posi // w   # = posi
    jj = posi % w    # = 0
    return li, fi, ii, jj, posi


assert n_orig_imgs // NDIRS == len(categories)

tensorX = np.zeros((n_neurons_to_pick, len(categories), NDIRS, n_shifts))
neurons_used = np.empty((n_neurons_to_pick, 5), dtype='int')

for nii, ni in enumerate(sampled_neurons):
    li, fi, ii, jj, posi = get_neuron_pos(ni)
    neurons_used[nii] = [li, fi, ii, jj, posi]
    for cati in range(len(categories)):
        pst = all_per_img_output[cati * NDIRS: (cati + 1) * NDIRS, fi, posi, :]
        tensorX[nii, cati] = pst  # (NDIRS, n_shifts)

print(f'tensorX shape    (N, S, D, T): {tensorX.shape}')
print(f'neurons_used shape:            {neurons_used.shape}')

# Construct output filename from CONFIG
models_str = "+".join(MODEL_IDXS)
SUFFIX = f"flyvis_{GROUP_NAME}_i{N_INSTANCES}_n{n_neurons_to_pick}_model{models_str}"
if seed > 0:
    SUFFIX += f"_seed{seed}"
print(f'\nOutput suffix: {SUFFIX}')

os.makedirs('../data/sampled', exist_ok=True)
out_tensor  = f'../data/sampled/tensor4d_{SUFFIX}.npy'
out_neurons = f'../data/sampled/neurons_used_{SUFFIX}.npy'

if os.path.exists(out_tensor) or os.path.exists(out_neurons):
    print("\nWARNING: output files already exist — delete them to overwrite.")
else:
    np.save(out_tensor, tensorX)
    print(f'Saved: {out_tensor}')
    np.save(out_neurons, neurons_used)
    print(f'Saved: {out_neurons}')